# NB03 — Features and the First Model
**Machine Learning in Finance · Week 3**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB03_features_first_model.ipynb)

**Finance question:** Does a stock's past tell you anything about its future?

**Key idea:** A **feature** is anything about a stock that you know *before* the period you're predicting. A model is a line (or curve) of best guess. We **train on the past and test on the future**.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Step 1: features
Five classic features that finance research has studied for decades:

| feature | what it measures | finance idea |
|---|---|---|
| `mom_12_1` | return over the past year, skipping last month | momentum: winners keep winning? |
| `rev_1` | last month's return | short-term reversal: last month's losers bounce? |
| `size` | log of market cap | small firms riskier, maybe higher return |
| `btm` | book value ÷ market value | value: "cheap" stocks |
| `vol_12` | volatility of the last 12 monthly returns | risk |

Each row's features use only information public at that month-end. The target `ret_next` is what
happened over the **following** month.

In [ ]:
panel = mlfin.make_features()
FEATURES = ["mom_12_1", "rev_1", "size", "btm", "vol_12"]
panel[["date", "ticker"] + FEATURES + ["ret_next"]].head()

## Step 2: one feature against next month's return

In [ ]:
# 🟡 YOUR TURN
SCATTER_FEATURE = "mom_12_1"   # ← try "rev_1", "size", "btm", "vol_12"
slope = mlfin.plot_scatter(panel, SCATTER_FEATURE)
print(f"Slope of the line: {slope:.4f}")

*What to look for:* a huge cloud and an almost flat red line. Next month's return is mostly noise;
any one feature explains very little of it. This is normal in finance — and why careful testing matters.

## Step 3: split the data by time
The model may only learn from the **past** (training data) and is graded on the **future** (test data)
— exactly the situation of an investor. We never shuffle months together.

In [ ]:
# 🟡 YOUR TURN
SPLIT_DATE = "2017-12-31"   # ← try "2014-12-31" or "2020-12-31"

train, test = mlfin.time_split(panel, SPLIT_DATE)
mlfin.plot_timeline(SPLIT_DATE)
print(f"Training: {len(train):,} stock-months   |   Testing: {len(test):,} stock-months")

## Step 4: fit the line of best guess (linear regression)

In [ ]:
# 🟡 YOUR TURN
FEATURES = ["mom_12_1", "rev_1", "size", "btm", "vol_12"]
# ← remove one, or add one of: "ep", "growth", "leverage", "beta_12", "max_1"
#   (mlfin.FEATURE_DESCRIPTIONS explains each)

In [ ]:
from sklearn.linear_model import LinearRegression
X_train, y_train, _ = mlfin.prepare_xy(train, FEATURES, "ret_next")
X_test, y_test, test_rows = mlfin.prepare_xy(test, FEATURES, "ret_next")
model = LinearRegression().fit(X_train, y_train)
pd.Series(model.coef_, index=FEATURES, name="weight in the line").round(4).to_frame()

*How to read the weights:* a positive weight means "more of this feature → the model predicts a higher
return". Don't compare sizes across features yet: they're measured in different units (Week 4 fixes that).

## Step 5: how good is it?
**R-squared** is the share of the ups and downs in returns that the model explains: 1 = perfectly, 0 = not
at all. We measure it on the training data (in-sample) and the test data (out-of-sample).

In [ ]:
print(f"R-squared, training data (in-sample):    {model.score(X_train, y_train):.4f}")
print(f"R-squared, test data   (out-of-sample):  {model.score(X_test, y_test):.4f}")

Both numbers are **below 1%**. Stock returns are mostly unpredictable noise, so a model can be useful
even with a tiny R-squared — *if* its predictions put stocks in roughly the right order. The next chart
checks exactly that.

## Step 6: do higher predictions come true?

In [ ]:
mlfin.plot_deciles(model.predict(X_test), y_test, dates=test_rows["date"],
                   title="Test period: average next-month return by prediction group")

*What to look for:* each month, stocks are sorted into 10 groups by the model's prediction. If the
model is useful, bars should rise from left (lowest predictions) to right (highest). They don't need to rise
smoothly — an investor mainly cares whether group 10 beats group 1.

In [ ]:
pred = pd.Series(model.predict(X_test), index=X_test.index)
groups = test_rows.assign(pred=pred).groupby("date")["pred"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 10, labels=False))
top = y_test[groups == 9].groupby(test_rows["date"]).mean()
bottom = y_test[groups == 0].groupby(test_rows["date"]).mean()
print(f"Top group beat bottom group by {(top - bottom).mean() * 100:.2f}% a month "
      f"({(top - bottom).mean() * 1200:.1f}% a year) in the test period,")
print(f"and did so in {((top - bottom) > 0).mean():.0%} of months.")

## ⭐ GOING FURTHER (optional)
1. Ask an AI assistant: *"Why can a stock-return model with an R-squared of 0.5% still be useful for investing?"*
   Do you agree with its answer?
2. Change `SPLIT_DATE` to `"2014-12-31"` and re-run Steps 3–6. Does the model trained on less data do worse?

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*